In [0]:
from pyspark.sql.functions import *

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS fraud_platform.feature;

In [0]:
%sql
CREATE VOLUME IF NOT EXISTS fraud_platform.feature.feature_volume;

In [0]:
silver_df = (
    spark.readStream
    .table("fraud_platform.silver.transactions_clean")
)

In [0]:
print(silver_df.isStreaming)

True


In [0]:
feature_df = (
    silver_df

    # High Value Transaction
    .withColumn(
        "high_value_transaction",
        when(col("TransactionAmt") >= 500, 1).otherwise(0)
    )

    # Amount Bucket
    .withColumn(
        "amount_bucket",
        when(col("TransactionAmt") < 50, "LOW")
        .when(col("TransactionAmt") < 200, "MEDIUM")
        .otherwise("HIGH")
    )

    # Email Match
    .withColumn(
        "email_match",
        when(col("P_emaildomain") == col("R_emaildomain"), 1).otherwise(0)
    )

    # Address Missing
    .withColumn(
        "address_missing",
        when(
            col("addr1").isNull() | col("addr2").isNull(),
            1
        ).otherwise(0)
    )

    # Credit Card Flag
    .withColumn(
        "credit_card",
        when(col("card6") == "credit", 1).otherwise(0)
    )

    # Debit Card Flag
    .withColumn(
        "debit_card",
        when(col("card6") == "debit", 1).otherwise(0)
    )

    # Visa Flag
    .withColumn(
        "is_visa",
        when(col("card4") == "visa", 1).otherwise(0)
    )

    # Mastercard Flag
    .withColumn(
        "is_mastercard",
        when(col("card4") == "mastercard", 1).otherwise(0)
    )

    # Processing Timestamp
    .withColumn(
        "feature_timestamp",
        current_timestamp()
    )
)

In [0]:
query = (
    feature_df.writeStream
    .format("delta")
    .outputMode("append")
    .trigger(availableNow=True)
    .option(
        "checkpointLocation",
        "/Volumes/fraud_platform/feature/feature_volume/checkpoints/features"
    )
    .toTable("fraud_platform.feature.transaction_features")
)

query.awaitTermination()
print("Feature Engineering completed")

Feature Engineering completed


In [0]:
%sql
SELECT *
FROM fraud_platform.feature.transaction_features
LIMIT 20;

TransactionID,isFraud,TransactionDT,TransactionAmt,ProductCD,card4,card6,addr1,addr2,P_emaildomain,R_emaildomain,ingestion_timestamp,event_date,high_value_transaction,amount_bucket,email_match,address_missing,credit_card,debit_card,is_visa,is_mastercard,feature_timestamp
2987001,0,86401,29.0,W,mastercard,credit,325.0,87.0,gmail.com,null,2026-07-04T14:26:07.698Z,2026-07-04,0,LOW,0,0,1,0,0,1,2026-07-04T14:47:00.581Z
2987003,0,86499,50.0,W,mastercard,debit,476.0,87.0,yahoo.com,null,2026-07-04T14:26:07.698Z,2026-07-04,0,MEDIUM,0,0,0,1,0,1,2026-07-04T14:47:00.581Z
2987005,0,86510,49.0,W,visa,debit,272.0,87.0,gmail.com,null,2026-07-04T14:26:07.698Z,2026-07-04,0,LOW,0,0,0,1,1,0,2026-07-04T14:47:00.581Z
2987007,0,86529,422.5,W,visa,debit,325.0,87.0,mail.com,null,2026-07-04T14:26:07.698Z,2026-07-04,0,HIGH,0,0,0,1,1,0,2026-07-04T14:47:00.581Z
2987011,0,86555,16.495,C,mastercard,debit,null,null,hotmail.com,hotmail.com,2026-07-04T14:26:07.698Z,2026-07-04,0,LOW,1,1,0,1,0,1,2026-07-04T14:47:00.581Z
2987013,0,86585,40.0,W,visa,debit,330.0,87.0,aol.com,null,2026-07-04T14:26:07.698Z,2026-07-04,0,LOW,0,0,0,1,1,0,2026-07-04T14:47:00.581Z
2987016,0,86620,30.0,H,visa,debit,170.0,87.0,aol.com,null,2026-07-04T14:26:07.698Z,2026-07-04,0,LOW,0,0,0,1,1,0,2026-07-04T14:47:00.581Z
2987018,0,86725,47.95,W,visa,debit,184.0,87.0,gmail.com,null,2026-07-04T14:26:07.698Z,2026-07-04,0,LOW,0,0,0,1,1,0,2026-07-04T14:47:00.581Z
2987021,0,86769,159.95,W,mastercard,debit,204.0,87.0,gmail.com,null,2026-07-04T14:26:07.698Z,2026-07-04,0,MEDIUM,0,0,0,1,0,1,2026-07-04T14:47:00.581Z
2987024,0,86821,73.95,W,visa,debit,264.0,87.0,gmail.com,null,2026-07-04T14:26:07.698Z,2026-07-04,0,MEDIUM,0,0,0,1,1,0,2026-07-04T14:47:00.581Z


In [0]:
%sql
SELECT COUNT(*)
FROM fraud_platform.feature.transaction_features;

COUNT(*)
40000
